### Step 1: Ingestion & Environment Setup
- **Objective:** Mount Google Drive and load the cleaned dataset provided by Farah (`data_cleaned final file.csv`) directly into the environment.

In [78]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [79]:
import pandas as pd

df = pd.read_csv('data_cleaned final file.csv')

print("Data Shape:", df.shape)
df.head()

Data Shape: (1462200, 19)


,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses,segment,ground_truth,window_id
0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0,0,0
1,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0,0,0
2,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0,0,0
3,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0,0,0
4,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0,0,0


### Step 2: Data Loading & Timestamp Parsing
- **Objective:** Parse ISO-8601 timestamps and verify dataset timeline boundaries.

In [80]:
import pandas as pd

data_path = 'data_cleaned final file.csv'
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Data Shape:", df.shape)
print(df['timestamp'].min())
print(df['timestamp'].max())

Data Shape: (1462200, 19)
2020-02-01 00:00:00
2020-09-01 03:56:32


In [81]:
data_path = 'data_cleaned final file.csv'
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(len(df))
print(df['timestamp'].min())
print(df['timestamp'].max())

1462200
2020-02-01 00:00:00
2020-09-01 03:56:32


### Step 3: Feature & Metadata Separation
- **Objective:** Separate raw sensor features (`X_features`) from metadata columns.

In [82]:
excluded_cols = ['timestamp', 'segment', 'ground_truth', 'window_id']
sensor_cols = [col for col in df.columns if col not in excluded_cols]

X_features = df[sensor_cols].copy()
timestamp_index = df[excluded_cols].copy()

print(X_features.shape)
print(timestamp_index.shape)

(1462200, 15)
(1462200, 4)


### Step 4: Event-Based Temporal Splitting
- **Objective:** Split data into train, validation, and test sets based on failure event cutoffs, ensuring zero failure leakage in the training set.

In [83]:
import numpy as np

cutoff_train_val = '2020-06-01 00:00:00'
cutoff_val_test = '2020-07-01 00:00:00'

conditions = [
    timestamp_index['timestamp'] < cutoff_train_val,
    (timestamp_index['timestamp'] >= cutoff_train_val)
    & (timestamp_index['timestamp'] < cutoff_val_test),
    timestamp_index['timestamp'] >= cutoff_val_test,
]

splits = ['train', 'validation', 'test']
timestamp_index['split'] = np.select(conditions, splits, default='test')

print(timestamp_index['split'].value_counts())

split
train         831360
test          437957
validation    192883
Name: count, dtype: int64


In [84]:
train_max = timestamp_index[timestamp_index['split'] == 'train'][
    'timestamp'
].max()
val_min = timestamp_index[timestamp_index['split'] == 'validation'][
    'timestamp'
].min()
val_max = timestamp_index[timestamp_index['split'] == 'validation'][
    'timestamp'
].max()
test_min = timestamp_index[timestamp_index['split'] == 'test'][
    'timestamp'
].min()

assert train_max < val_min < val_max < test_min
assert len(X_features) == len(df)
assert len(timestamp_index) == len(df)
assert X_features.isna().sum().sum() == 0

print("All assertions passed successfully.")

All assertions passed successfully.


In [85]:
import json
import os

output_dir = '/content/phase2_task1_outputs'
os.makedirs(output_dir, exist_ok=True)

X_features.to_csv(f'{output_dir}/X_features.csv', index=False)
timestamp_index.to_csv(f'{output_dir}/timestamp_index.csv', index=False)

config = {
    'ordered_features': sensor_cols,
    'excluded_columns': excluded_cols,
    'cutoff_timestamps': {
        'train_val': cutoff_train_val,
        'val_test': cutoff_val_test,
    },
    'failures_distribution': {
        'train': [1, 2],
        'validation': [3],
        'test': [4],
    },
    'total_rows': len(df),
    'splits_count': timestamp_index['split'].value_counts().to_dict(),
}

with open(f'{output_dir}/features_config.json', 'w') as f:
  json.dump(config, f, indent=4)

print("Files saved successfully.")

Files saved successfully.


In [86]:
report_content =rf"""# Phase 2 - Task 1: Feature Separation & Event-Based Split Report

## 1. Overview & Data Summary
- **Total Samples:** {len(df):,}
- **Time Range:** {df['timestamp'].min()} to {df['timestamp'].max()}
- **Sensor Features Count:** {X_features.shape[1]}
- **Metadata Columns:** {excluded_cols}

---

## 2. Cutoff Timestamps & Split Logic
Splits were constructed based on historical failure events to prevent temporal leakage and evaluate model performance across unseen failures:
- **Cutoff 1 (Train / Validation):** `{cutoff_train_val}`
- **Cutoff 2 (Validation / Test):** `{cutoff_val_test}`

---

## 3. Failure Events Distribution
- **Train Set:** Event 1 (2020-04-18) and Event 2 (2020-05-29 to 2020-05-30)
- **Validation Set:** Event 3 (2020-06-05 to 2020-06-07)
- **Test Set:** Event 4 (2020-07-15)

---

## 4. Split Statistics
| Split | Row Count | Percentage |
| :--- | :--- | :--- |
| **Train** | 856,832 | 56.48% |
| **Validation** | 216,514 | 14.27% |
| **Test** | 443,602 | 29.24% |
| **Total** | **1,516,948** | **100.00%** |

---

## 5. Verification & Acceptance Criteria
- **Temporal Order:** Verified ($\max(Train) < \min(Val) < \min(Test)$)[cite: 7].
- **Data Integrity:** No missing values in sensor features, row counts strictly match source dataset[cite: 7].
"""

with open(f"{output_dir}/split_report.md", "w") as f:
    f.write(report_content)

print("split_report.md created successfully.")

split_report.md created successfully.


In [87]:
import shutil
from google.colab import files

shutil.make_archive('phase2_task1_outputs', 'zip', '/content/phase2_task1_outputs')
files.download('phase2_task1_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [88]:
print(df.groupby('segment')['timestamp'].agg(['min', 'max', 'count']))

                        min                 max  count
segment                                               
0       2020-02-01 00:00:00 2020-02-01 12:48:00   4650
1       2020-02-01 12:53:47 2020-02-01 19:40:04   2460
2       2020-02-01 23:15:33 2020-02-01 23:20:21     30
3       2020-02-02 04:34:27 2020-02-02 07:37:40   1110
4       2020-02-02 07:41:52 2020-02-05 00:32:21  23550
...                     ...                 ...    ...
327     2020-08-30 04:26:10 2020-08-31 00:30:23   7290
328     2020-08-31 01:49:32 2020-08-31 06:36:50   1740
329     2020-08-31 06:42:31 2020-08-31 20:15:10   4920
330     2020-09-01 00:37:33 2020-09-01 00:42:20     30
331     2020-09-01 02:37:24 2020-09-01 03:56:32    480

[325 rows x 3 columns]


In [89]:
excluded_cols = ['timestamp', 'segment', 'ground_truth', 'window_id']
sensor_cols = [col for col in df.columns if col not in excluded_cols]

X_features = df[sensor_cols].copy()
timestamp_index = df[excluded_cols].copy()

print(X_features.shape)
print(timestamp_index.shape)

(1462200, 15)
(1462200, 4)


### Step 5: Event-Based Temporal Splitting
- **Objective:** Split data into train, validation, and test sets based on failure event cutoff timestamps, ensuring zero failure leakage in the training set (`ground_truth = 0` only).

In [90]:
import numpy as np

cutoff_train_val = '2020-04-12 00:00:00'
cutoff_val_test = '2020-07-01 00:00:00'

conditions = [
    timestamp_index['timestamp'] < cutoff_train_val,
    (timestamp_index['timestamp'] >= cutoff_train_val) & (timestamp_index['timestamp'] < cutoff_val_test),
    timestamp_index['timestamp'] >= cutoff_val_test
]

splits = ['train', 'validation', 'test']
timestamp_index['split'] = np.select(conditions, splits, default='test')

train_idx = timestamp_index['split'] == 'train'
leakage_mask = train_idx & (timestamp_index['ground_truth'] != 0)
timestamp_index.loc[leakage_mask, 'split'] = 'validation'

print(timestamp_index['split'].value_counts())

split
train         521898
validation    502345
test          437957
Name: count, dtype: int64


### Step 5: Verification & Self-Checks
- **Objective:** Run programmatic assertions to verify temporal ordering, check data integrity, and ensure zero missing values.

In [91]:
train_max = timestamp_index[timestamp_index['split'] == 'train']['timestamp'].max()
val_min = timestamp_index[timestamp_index['split'] == 'validation']['timestamp'].min()
val_max = timestamp_index[timestamp_index['split'] == 'validation']['timestamp'].max()
test_min = timestamp_index[timestamp_index['split'] == 'test']['timestamp'].min()

assert train_max < val_min < val_max < test_min
assert len(X_features) == len(df)
assert len(timestamp_index) == len(df)
assert X_features.isna().sum().sum() == 0

print("All assertions passed successfully.")

All assertions passed successfully.


### Step 6: Exporting Outputs & Configuration
- **Objective:** Save processed features (`X_features.csv`), metadata indices (`timestamp_index.csv`), update configuration (`features_config.json` with `random_state = 42`), and generate the final split report.

In [92]:
import json
import os

output_dir = '/content/phase2_task1_outputs'
os.makedirs(output_dir, exist_ok=True)

X_features.to_csv(f'{output_dir}/X_features.csv', index=False)
timestamp_index.to_csv(f'{output_dir}/timestamp_index.csv', index=False)

train_count = int((timestamp_index['split'] == 'train').sum())
val_count = int((timestamp_index['split'] == 'validation').sum())
test_count = int((timestamp_index['split'] == 'test').sum())
total_count = len(df)

config = {
    "ordered_features": sensor_cols,
    "excluded_columns": excluded_cols,
    "cutoff_timestamps": {
        "train_val": cutoff_train_val,
        "val_test": cutoff_val_test
    },
    "failures_distribution": {
        "train": [1, 2],
        "validation": [3],
        "test": [4]
    },
    "total_rows": total_count,
    "splits_count": {
        "train": train_count,
        "validation": val_count,
        "test": test_count
    },
    "random_state": 42
}

with open(f'{output_dir}/features_config.json', 'w') as f:
    json.dump(config, f, indent=4)

report_content = rf"""# Phase 2 - Task 1: Feature Separation & Event-Based Split Report

## 1. Overview & Data Summary
- **Total Samples (After Cleaning):** {total_count:,}
- **Time Range:** {df['timestamp'].min()} to {df['timestamp'].max()}
- **Sensor Features Count:** {X_features.shape[1]}
- **Metadata Columns:** {excluded_cols}

---

## 2. Cutoff Timestamps & Split Logic
Splits were constructed based on historical failure events:
- **Cutoff 1 (Train / Validation):** `{cutoff_train_val}`
- **Cutoff 2 (Validation / Test):** `{cutoff_val_test}`

---

## 3. Failure Events Distribution
- **Train Set:** Event 1 (2020-04-18) and Event 2 (2020-05-29 to 2020-05-30)
- **Validation Set:** Event 3 (2020-06-05 to 2020-06-07)
- **Test Set:** Event 4 (2020-07-15)

---

## 4. Split Statistics
| Split | Row Count | Percentage |
| :--- | :--- | :--- |
| **Train** | {train_count:,} | {train_count / total_count * 100:.2f}% |
| **Validation** | {val_count:,} | {val_count / total_count * 100:.2f}% |
| **Test** | {test_count:,} | {test_count / total_count * 100:.2f}% |
| **Total** | **{total_count:,}** | **100.00%** |

---

## 5. Verification & Acceptance Criteria
- **Temporal Order:** Verified ($\max(Train) < \min(Val) < \min(Test)$).
- **Data Integrity:** No missing values in sensor features, row counts match cleaned dataset.
"""

with open(f"{output_dir}/split_report.md", "w") as f:
    f.write(report_content)

print("Files saved successfully.")

Files saved successfully.


In [93]:
import shutil
from google.colab import files

shutil.make_archive('phase2_task1_cleaned_outputs', 'zip', '/content/phase2_task1_outputs')
files.download('phase2_task1_cleaned_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [94]:
import pandas as pd

ts_df = pd.read_csv('/content/phase2_task1_outputs/timestamp_index.csv')
train_failures = ts_df[(ts_df['split'] == 'train') & (ts_df['ground_truth'] != 0)]

print(f"Number of leaked failure rows in train: {len(train_failures)}")
assert len(train_failures) == 0, "AssertionError: Training set contains failure rows!"
print("Self-check passed successfully!")

Number of leaked failure rows in train: 0
Self-check passed successfully!


In [95]:
train_max_date = ts_df[ts_df['split'] == 'train']['timestamp'].max()
test_min_date = ts_df[ts_df['split'] == 'test']['timestamp'].min()
assert train_max_date < test_min_date
print("Temporal split check passed.")

Temporal split check passed.


In [96]:
x_df = pd.read_csv('/content/phase2_task1_outputs/X_features.csv')
assert len(x_df) == len(ts_df)
assert x_df.isna().sum().sum() == 0
print("Data integrity check passed.")

Data integrity check passed.


In [97]:
import json
import os

output_dir = '/content/phase2_task1_outputs'
os.makedirs(output_dir, exist_ok=True)

X_features.to_csv(f'{output_dir}/X_features.csv', index=False)
timestamp_index.to_csv(f'{output_dir}/timestamp_index.csv', index=False)

train_count = int((timestamp_index['split'] == 'train').sum())
val_count = int((timestamp_index['split'] == 'validation').sum())
test_count = int((timestamp_index['split'] == 'test').sum())
total_count = len(df)

config = {
    "ordered_features": sensor_cols,
    "excluded_columns": excluded_cols,
    "cutoff_timestamps": {
        "train_val": cutoff_train_val,
        "val_test": cutoff_val_test
    },
    "failures_distribution": {
        "train": [1, 2],
        "validation": [3],
        "test": [4]
    },
    "total_rows": total_count,
    "splits_count": {
        "train": train_count,
        "validation": val_count,
        "test": test_count
    },
    "random_state": 42
}

with open(f'{output_dir}/features_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print("Config updated successfully.")

Config updated successfully.


### Step 7: Data Summary & Integrity Check
- **Objective:** Verify total row/column counts, check for missing values, and ensure zero duplicate rows in the dataset.

In [98]:
import pandas as pd

df = pd.read_csv('data_cleaned final file.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"--- 1. Data Summary Check ---")
print(f"Total Rows & Columns: {df.shape}")
print(f"Missing Values Count: {df.isna().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")

--- 1. Data Summary Check ---
Total Rows & Columns: (1462200, 19)
Missing Values Count: 0
Duplicate Rows: 0


### Step 8: Frozen Periods Quick Check
- **Objective:** Audit sensor features for any prolonged static or frozen blocks.

In [99]:
print(f"\n--- 2. Frozen Periods Quick Check ---")
sensor_cols = [c for c in df.columns if c not in ['timestamp', 'segment', 'ground_truth', 'window_id']]
frozen_found = False

for col in sensor_cols:
    is_col_frozen = (df[col].diff() == 0)
    groups = (~is_col_frozen).cumsum()
    counts = is_col_frozen.groupby(groups).sum()
    if len(counts) > 0 and counts.max() > 500:
        print(f"Notice: Sensor {col} has a static block of {counts.max()} rows.")
        frozen_found = True

if not frozen_found:
    print("Check Passed: No abnormal long frozen periods detected in sensor features!")


--- 2. Frozen Periods Quick Check ---
Notice: Sensor COMP has a static block of 17874 rows.
Notice: Sensor DV_eletric has a static block of 17874 rows.
Notice: Sensor MPG has a static block of 17874 rows.
Notice: Sensor LPS has a static block of 253615 rows.
Notice: Sensor Pressure_switch has a static block of 7274 rows.
Notice: Sensor Oil_level has a static block of 825863 rows.
Notice: Sensor Caudal_impulses has a static block of 1168919 rows.


### Step 9: Temporal Split & Leakage Verification
- **Objective:** Run final programmatic assertions to guarantee zero failure leakage in the training set and verify strict temporal ordering.

In [100]:
import pandas as pd

ts_df = pd.read_csv('/content/phase2_task1_outputs/timestamp_index.csv')


train_failures = ts_df[(ts_df['split'] == 'train') & (ts_df['ground_truth'] != 0)]
print(f"\n--- 3. Leakage & Split Check ---")
print(f"Leaked failure rows in train set: {len(train_failures)}")
assert len(train_failures) == 0, "AssertionError: Training set contains failure rows!"


train_max = ts_df[ts_df['split'] == 'train']['timestamp'].max()
val_min = ts_df[ts_df['split'] == 'validation']['timestamp'].min()
val_max = ts_df[ts_df['split'] == 'validation']['timestamp'].max()
test_min = ts_df[ts_df['split'] == 'test']['timestamp'].min()

print(f"Train Max Date: {train_max}")
print(f"Validation Min/Max: {val_min} to {val_max}")
print(f"Test Min Date: {test_min}")

assert train_max < val_min < val_max < test_min, "AssertionError: Temporal order is violated!"
print("All temporal split and integrity assertions passed successfully!")


--- 3. Leakage & Split Check ---
Leaked failure rows in train set: 0
Train Max Date: 2020-04-11 23:59:51
Validation Min/Max: 2020-04-12 00:00:01 to 2020-06-30 23:59:58
Test Min Date: 2020-07-01 00:00:08
All temporal split and integrity assertions passed successfully!
